# ADM1-Benchmark: laden · ansehen · nutzen

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from loader import CHANNELS, load_meta, load_test, load_train
from scoring import score_dataset

## 1 · Laden

In [ ]:
train = load_train()
test  = load_test()
meta  = load_meta()

Form der Trainings-Arrays (100 Reihen × 1441 Stunden = 60 Tage):

In [ ]:
{k: train[k].shape for k in ['measurements', 'feed_noisy', 'states']}

Testset (Reihen unterschiedlich lang → Liste von Dicts):

In [ ]:
import collections

print(len(test), 'Testreihen · Modi:', dict(collections.Counter(s['regime'] for s in test)))
{k: np.shape(test[0][k]) for k in ['measurements', 'states', 'ukf_x_hat']}

**UKF-Referenz:** `ukf_x_hat` / `ukf_std` sind im Datensatz enthalten. Sie stammen aus dem mit CMA-ES abgestimmten vollen UKF, die Parameter dazu stehen in `ukf_calibration.json`. Falls `ukf_pending=True` gesetzt ist, fehlt die Referenz fuer diese Reihe und die UKF-Spalten im Scoring erscheinen als `nan`, die eigenen Werte sind davon unberuehrt.

## 2 · Datensatz ansehen

Die 5 Sensoren und ihr Rauschen:

In [ ]:
pd.Series(meta['sensor_noise'])

Die 5 Substrate:

In [ ]:
pd.DataFrame([{'Substrat': k, 'Fluss': v['nominal_flow_m3_per_d'],
               'TS': v['characterisation']['TS'], 'BMP': v['characterisation']['BMP']}
              for k, v in meta['substrates'].items()])

In [ ]:
s = test[0]; t = s['time']
fig, ax = plt.subplots(1, 5, figsize=(16, 3))
for i, ch in enumerate(CHANNELS):
    ax[i].plot(t, s['measurements'][:, i]); ax[i].set_title(ch)
fig.tight_layout()

Substrat-Feed mit den Wechselzeiten:

Ein versteckter Zustand (S_ac) — Wahrheit vs. UKF-Referenz (nur falls UKF schon berechnet):

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.5))
for j, name in enumerate(meta['substrates']):
    ax.plot(t, s['feed_noisy'][:, j], label=name)
for d in np.atleast_1d(s['switch_days']):
    ax.axvline(float(d), color='0.7', ls=':', lw=0.8)
ax.set_xlabel('Tage'); ax.set_ylabel('Feed [m³/d]'); ax.legend(fontsize=8)
fig.tight_layout()

In [ ]:
plt.figure(figsize=(10, 3.5))
plt.plot(t, s['states'][:, 6], 'k', label='Truth')
if not s.get('ukf_pending', False):
    plt.plot(t, s['ukf_x_hat'][:, 6], 'r', label='UKF')
plt.title('S_ac (idx 6)'); plt.xlabel('Tage'); plt.legend()

## 3 · Baseline: Messungen + Feed → Zustände

Merkmale und Ziel aus allen Trainingsreihen zusammenlegen:

In [ ]:
X = np.concatenate([train['measurements'], train['feed_noisy']], -1).reshape(-1, 10)
Y = train['states'].reshape(-1, 41)

Lineare Abbildung per Least-Squares fitten:

In [ ]:
mu, sd = X.mean(0), X.std(0) + 1e-9
W, *_ = np.linalg.lstsq(np.c_[(X - mu) / sd, np.ones(len(X))], Y, rcond=None)

Vorhersage-Funktion und auf alle Testreihen anwenden:

In [ ]:
def model(s):
    Xs = np.concatenate([s['measurements'], s['feed_noisy']], -1)
    return np.c_[(Xs - mu) / sd, np.ones(len(Xs))] @ W   # -> (T, 41)

pred = [model(s) for s in test]

## 4 · Scoren gegen den UKF

In [ ]:
r = score_dataset(pred, test)
pd.DataFrame({'Baseline': r['model_mean'], 'UKF': r['ukf_reference_mean']}).round(1)

Kopfmetrik ist die **Transienten-NRMSE** (Median, kleiner = besser). Ziel: den UKF schlagen.